# Лабораторная работа 2.2(R)
# Выполнил: Хрипков Тимофей, группа ИУ5-65Б
#
# Обработка пропусков в данных, кодирование категориальных признаков и масштабирование

In [ ]:
install.packages(c("palmerpenguins", "dplyr", "tidyr", "ggplot2", "caret", "fastDummies"))

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘listenv’, ‘parallelly’, ‘future’, ‘globals’, ‘shape’, ‘future.apply’, ‘numDeriv’, ‘progressr’, ‘SQUAREM’, ‘diagram’, ‘lava’, ‘prodlim’, ‘proxy’, ‘iterators’, ‘clock’, ‘gower’, ‘hardhat’, ‘ipred’, ‘sparsevctrs’, ‘timeDate’, ‘e1071’, ‘foreach’, ‘ModelMetrics’, ‘plyr’, ‘pROC’, ‘recipes’, ‘reshape2’




Я буду использовать готовый датасет Palmer Penguins


In [ ]:
library(palmerpenguins)
library(dplyr)
library(tidyr)
library(ggplot2)
library(caret)
library(fastDummies)

[1] "Библиотеки успешно загружены"


In [ ]:
df <- penguins
cat("Размер:", nrow(df), "x", ncol(df), "\n")
head(df)

Размер: 344 x 8 


species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
<fct>,<fct>,<dbl>,<dbl>,<int>,<int>,<fct>,<int>
Adelie,Torgersen,39.1,18.7,181,3750,male,2007
Adelie,Torgersen,39.5,17.4,186,3800,female,2007
Adelie,Torgersen,40.3,18.0,195,3250,female,2007
Adelie,Torgersen,NA,NA,NA,NA,NA,2007
Adelie,Torgersen,36.7,19.3,193,3450,female,2007
Adelie,Torgersen,39.3,20.6,190,3650,male,2007


In [ ]:
colSums(is.na(df))
cat("Всего пропусков:", sum(colSums(is.na(df))), "\n")

species            island    bill_length_mm     bill_depth_mm 
                0                 0                 2                 2 
flipper_length_mm       body_mass_g               sex              year 
                2                 2                11                 0

Всего пропусков: 19 


Строки с пропусками в колонке `sex` можно удалить, так как их немного.

In [ ]:
clear <- df %>% filter(!is.na(sex))
cat("Удалено строк:", nrow(df) - nrow(clear), "\n")
cat("Осталось строк:", nrow(clear), "\n")

Удалено строк: 11 
Осталось строк: 333 


In [ ]:
clear <- clear %>%
  mutate(
    bill_length_mm = ifelse(is.na(bill_length_mm), median(bill_length_mm, na.rm = TRUE), bill_length_mm),
    bill_depth_mm = ifelse(is.na(bill_depth_mm), median(bill_depth_mm, na.rm = TRUE), bill_depth_mm),
    flipper_length_mm = ifelse(is.na(flipper_length_mm), median(flipper_length_mm, na.rm = TRUE), flipper_length_mm),
    body_mass_g = ifelse(is.na(body_mass_g), median(body_mass_g, na.rm = TRUE), body_mass_g)
  )

colSums(is.na(clear))
cat("Пропусков нет\n")

species            island    bill_length_mm     bill_depth_mm 
                0                 0                 0                 0 
flipper_length_mm       body_mass_g               sex              year 
                0                 0                 0                 0

Пропусков нет


In [ ]:
cat("=== ПРОВЕРКА sex ===\n")
cat("Тип:", class(clear$sex), "\n")
cat("Уникальные значения:", paste(unique(clear$sex), collapse = ", "), "\n")
print(table(clear$sex))

=== ПРОВЕРКА sex ===
Тип: factor 
Уникальные значения: male, female 

female   male 
   165    168 


От пропусков избавились. Переходим к кодированию категориальных признаков.
Закодируем пол как 0 и 1.

In [ ]:
# Кодируем фактор: Male -> 0, Female -> 1
clear <- clear %>%
  mutate(sex = as.numeric(sex) - 1)

# Проверка
cat("=== ПОСЛЕ КОДИРОВАНИЯ ===\n")
print(table(clear$sex))
cat("Male (0):", sum(clear$sex == 0), "\n")
cat("Female (1):", sum(clear$sex == 1), "\n")
cat("Всего:", nrow(clear), "\n")

=== ПОСЛЕ КОДИРОВАНИЯ ===

  0   1 
165 168 
Male (0): 165 
Female (1): 168 
Всего: 333 


Вид пингвина (`species`) мы закодируем от 1 до 3, так как его можно упорядочить.

In [ ]:
clear <- clear %>%
  mutate(species = as.numeric(species))

cat("=== КОДИРОВАНИЕ SPECIES ===\n")
print(table(clear$species))
cat("Adelie = 1, Chinstrap = 2, Gentoo = 3\n")

=== КОДИРОВАНИЕ SPECIES ===

  1   2   3 
146  68 119 
Adelie = 1, Chinstrap = 2, Gentoo = 3


Закодируем one-hot колонку `island`

In [ ]:
clear <- dummy_cols(clear, select_columns = "island", remove_selected_columns = TRUE)

clear <- clear %>%
  mutate(
    island_Biscoe = as.integer(island_Biscoe),
    island_Dream = as.integer(island_Dream),
    island_Torgersen = as.integer(island_Torgersen)
  )

cat("One-hot кодирование выполнено\n")

ERROR: Error in dummy_cols(clear, select_columns = "island", remove_selected_columns = TRUE): select_columns is/are not in data. Please check data and spelling.


In [ ]:
cat("=== СТРУКТУРА ===\n")
str(clear)
cat("\nКолонки:", paste(colnames(clear), collapse = ", "), "\n")

=== СТРУКТУРА ===
tibble [333 × 10] (S3: tbl_df/tbl/data.frame)
 $ species          : num [1:333] 1 1 1 1 1 1 1 1 1 1 ...
 $ bill_length_mm   : num [1:333] 39.1 39.5 40.3 36.7 39.3 38.9 39.2 41.1 38.6 34.6 ...
 $ bill_depth_mm    : num [1:333] 18.7 17.4 18 19.3 20.6 17.8 19.6 17.6 21.2 21.1 ...
 $ flipper_length_mm: int [1:333] 181 186 195 193 190 181 195 182 191 198 ...
 $ body_mass_g      : int [1:333] 3750 3800 3250 3450 3650 3625 4675 3200 3800 4400 ...
 $ sex              : num [1:333] 1 0 0 0 1 0 1 0 1 1 ...
 $ year             : int [1:333] 2007 2007 2007 2007 2007 2007 2007 2007 2007 2007 ...
 $ island_Biscoe    : int [1:333] 0 0 0 0 0 0 0 0 0 0 ...
 $ island_Dream     : int [1:333] 0 0 0 0 0 0 0 0 0 0 ...
 $ island_Torgersen : int [1:333] 1 1 1 1 1 1 1 1 1 1 ...

Колонки: species, bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g, sex, year, island_Biscoe, island_Dream, island_Torgersen 


Масштабируем данные.

In [ ]:
# Сохраняем категориальные и бинарные колонки
species_col <- clear$species
sex_col <- clear$sex
island_cols <- clear[, c("island_Biscoe", "island_Dream", "island_Torgersen")]

# Только непрерывные признаки для масштабирования
continuous_features <- clear %>%
  select(bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g)

# Масштабирование
scaler <- preProcess(continuous_features, method = "range")
continuous_scaled <- predict(scaler, continuous_features)

# Сборка финального датасета (ТОЛЬКО нужные колонки!)
clear_scaled <- cbind(
  species = species_col,
  continuous_scaled,
  sex = sex_col,
  island_cols
)

# Убеждаемся, что только 9 колонок
cat("Финальный датасет:", nrow(clear_scaled), "x", ncol(clear_scaled), "\n")
cat("Колонки:", paste(colnames(clear_scaled), collapse = ", "), "\n")

Финальный датасет: 333 x 9 
Колонки: species, bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g, sex, island_Biscoe, island_Dream, island_Torgersen 


In [ ]:
cat("=== ФИНАЛЬНАЯ ПРОВЕРКА ===\n")

# 1. Проверяем колонки
cat("\n1. Колонки (должно быть 9):\n")
print(colnames(clear_scaled))
cat("Количество:", ncol(clear_scaled), "\n")

# 2. Проверяем sex
cat("\n2. Распределение sex:\n")
print(table(clear_scaled$sex))
cat("Male (0):", sum(clear_scaled$sex == 0), "\n")
cat("Female (1):", sum(clear_scaled$sex == 1), "\n")

# 3. Проверяем species
cat("\n3. Распределение species:\n")
print(table(clear_scaled$species))

# 4. Проверяем диапазоны
cat("\n4. Диапазоны признаков:\n")
for(col in c("bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g")) {
  cat("  ", col, ":",
      "min =", round(min(clear_scaled[[col]]), 3),
      ", max =", round(max(clear_scaled[[col]]), 3), "\n")
}

cat("\n5. Бинарные признаки:\n")
for(col in c("sex", "island_Biscoe", "island_Dream", "island_Torgersen")) {
  cat("  ", col, ":",
      "min =", min(clear_scaled[[col]]),
      ", max =", max(clear_scaled[[col]]),
      ", уникальных значений:", length(unique(clear_scaled[[col]])), "\n")
}

# 5. Первые строки
cat("\n6. Первые 5 строк:\n")
print(head(clear_scaled, 5))

# 6. Итог
cat("\nВсе проверки пройдены!\n")
cat("Размер:", nrow(clear_scaled), "x", ncol(clear_scaled), "\n")

=== ФИНАЛЬНАЯ ПРОВЕРКА ===

1. Колонки (должно быть 9):
[1] "species"           "bill_length_mm"    "bill_depth_mm"    
[4] "flipper_length_mm" "body_mass_g"       "sex"              
[7] "island_Biscoe"     "island_Dream"      "island_Torgersen" 
Количество: 9 

2. Распределение sex:

  0   1 
165 168 
Male (0): 165 
Female (1): 168 

3. Распределение species:

  1   2   3 
146  68 119 

4. Диапазоны признаков:
   bill_length_mm : min = 0 , max = 1 
   bill_depth_mm : min = 0 , max = 1 
   flipper_length_mm : min = 0 , max = 1 
   body_mass_g : min = 0 , max = 1 

5. Бинарные признаки:
   sex : min = 0 , max = 1 , уникальных значений: 2 
   island_Biscoe : min = 0 , max = 1 , уникальных значений: 2 
   island_Dream : min = 0 , max = 1 , уникальных значений: 2 
   island_Torgersen : min = 0 , max = 1 , уникальных значений: 2 

6. Первые 5 строк:
  species bill_length_mm bill_depth_mm flipper_length_mm body_mass_g sex
1       1      0.2545455     0.6666667         0.1525424   0.2916667 